<a href="https://colab.research.google.com/github/sjoshi63/Story/blob/master/LLMRestService.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#!pip install Flask

###This should be a plain .py file (not a notebook)
###run this using the command : "python LLMRestService.py" (it will run on the specified port)

In [ ]:
from flask import Flask, jsonify, request
vectdb_dir = "./hcd/v3"

In [ ]:
app = Flask(__name__)

items = [
    {"id": 1, "name": "Item 1", "description": "this is item 1"},
    {"id": 2, "name": "Item 2", "description": "this is item 2"},
    {"id": 3, "name": "Item 3", "description": "this is item 3"},
]

In [ ]:
@app.route("/items/<int:item_id>", methods=["GET"])
def get_items(item_id):
    item = next((item for item in items if item["id"] == item_id), None)
    return jsonify(items)

In [ ]:
@app.route('/query', methods=['POST'])
def user_query():
    mystr = request.json
    tmp = mystr['user_input']
    print('user_data - ',tmp)
    qstr = f'''
Generate SQL to get data for the following query:
'''+tmp+f'''

Do not infer any data based on previous training, strictly use only source text below as input
'''
    print("query string -",qstr)
    llmresult = query_llm(llm, vectordb, qstr)
    print("LLM Result - ",llmresult)
    return jsonify(llmresult)


In [ ]:
@aoo.route('/items', methods=['GET'])
def get_items():
    return jsonify(items)

In [ ]:
def init_LLM():
import openai
import os
from azure.identity import CertificateCredential
import warnings
warnings.filterwarnings('ignore')
from langchain.chat_models import AzureChatOpenAI

  #==================================================================================================
  #.                     REQUIRED: Change these values
  #==================================================================================================
  client_id = "B52263DD-24A7-4F7C-8997-MYCLIENT-ID"
  certificate_path = "mycert.pem"
  base_url = "https:/llmopenai.mycompany.net/path/"
  ## Set Proxy
  os.environ["http_proxy"] = "proxy.jpmchase.net:8080"
  os.environ["https_proxy"] = "proxy.jpmchase.net:8080"
  if 'no_proxy' in os.environ:
    os.environ['no_proxy'] = os.environ['no_proxy'] + ',proxy.mycompany.net' + ',openai.azure.com'
  else:
    os.environ['no_proxy'] = 'localhost, 127.0.0.1,jpmchase.net,openai.azure.com'
  #==================================================================================================
  #                 GET OPEANAI Access Token
  #==================================================================================================
  scope = "https://cognitiveservices.azure.com/.default"
  credential = CertificateCredential(client_id=client_id, certificate_path=certificate_path,
                                     tenent_id='112abcd-25cd-4c36-mytenant-id', scope=scope)
  access_token = credential.get_token(scope).token
  #print(access_token)

  openai.api_type = "azure_ad"
  openai.api_base = base_url
  openai.api_key = access_token

  os.environ["OPENAI_API_TYPE"] = "azure_ad"
  os.environ["OPENAI_API_BASE"] = base_url
  os.environ["OPENAI_API_KEY"] = access_token
  os.environ["OPENAI_API_VERSION"] = "2023-03-15-preview"

  llm = AzureChatOpenAI(deployment_name="gpt-4-32k-0613", model_name='gpt-4-32k', temperature=0)

  return llm

In [ ]:
def get_embed():
  from langchain.vectorstores import Chroma
  from langchain.embeddings import OpenAIEmbeddings, AzureOpenAIEmbeddings
  from langchain.document_loaders import TextLoader, PyPDFLoader
  from langchain.text_splitter import CharacterTextSplitter, RecursiveCharacterTextSplitter
  import os
  import warnings
  warnings.filterwarnings('ignore')

  directory_path = "./documents"
  documents=[]
  for filename in os.listdir(directory_path):
    if filename.endswith(".txt"):
      loader = TextLoader(os.path.join(directory_path, filename))
      documents.extend(loader.load())

  embeddings = OpenAIEmbeddings(deployment="text-embedding-ada-002", chunk_size=1)
  text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
  docs = text_splitter.split_documents(documents)
  vectordb = Chroma.from_documents(documents=docs, embedding=embeddings, persist_directory=vectdb_dir)

  return vectordb

In [ ]:
def query_llm(llm, vectordb, query_string):
  from langchain.chains import RetrievalQA
  qa = RetrievalQA.from_chain_type(llm=llm, chain_type="stuff", retriever=vectordb.as_retriever())
  result = qa.run(query_string)
  return result

In [ ]:
llm = init_LLM()
import os
# Delete previously created embeddings if they exist
if os.path.isdir(vectdb_dir)
  os.rmdir(vectdb_dir)
#Create embeddings
vectordb = get_embed()

In [ ]:
#mystr = "how many error incidents were logged since the beginning of 2024?"

In [ ]:
# qstr = f'''

# '''+mystr+f'''

# Do not infer any data based on previous training, strictly use only source text below as input

# '''

In [ ]:
# result = query_llm(llm, vectordb, qstr)

In [ ]:
# print(result)